# Train Schedule Analysis and Interactive Route Enquiry System
### Sysslan IT Solutions — Internship Cohort | Data Analysis Project

**Dataset:** Indian Railway Train Schedule  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Levels:** 1 (Basic Review) → 2 (Processing) → 3 (Quality Checks) → 4 (Analysis & Viz) → 5 (Advanced) → 6 (Capstone Enquiry System)

## Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded successfully.")

## Load Dataset

In [ ]:
# Update path if needed
FILE_PATH = 'Dataset1.csv'

df = pd.read_csv(FILE_PATH,
                 usecols=['SN','Train_No','Station_Code','Station_Name',
                          'Route_Number','Arrival_time','Departure_Time','Distance'],
                 low_memory=False)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

---
## Level 1: Basic Data Review
> Goal: Understand dataset structure, train count, routes and stops.

In [ ]:
# Task 1.1 — Dataset overview
print("=" * 50)
print(f"Total Records  : {len(df):,}")
print(f"Attributes     : {len(df.columns)}")
print(f"Unique Trains  : {df['Train_No'].nunique():,}")
print(f"Unique Stations: {df['Station_Name'].nunique():,}")
print("=" * 50)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

In [ ]:
# Task 1.2 — List all trains with start and end stations
train_routes = df.groupby('Train_No').agg(
    Start_Station=('Station_Name', 'first'),
    End_Station=('Station_Name', 'last'),
    Total_Stops=('Station_Name', 'count')
).reset_index()

print(f"Total trains: {len(train_routes)}")
train_routes.head(15)

In [ ]:
# Task 1.3 — Number of stops per train
stops_per_train = df.groupby('Train_No')['Station_Name'].count().reset_index()
stops_per_train.columns = ['Train_No', 'Num_Stops']
print("Stops per train (sample):")
stops_per_train.head(10)

In [ ]:
# Task 1.4 — Train with max and min stops
max_stops_idx = stops_per_train['Num_Stops'].idxmax()
min_stops_idx = stops_per_train['Num_Stops'].idxmin()

max_row = stops_per_train.loc[max_stops_idx]
min_row = stops_per_train.loc[min_stops_idx]

print(f"Train with MOST stops  : Train {max_row['Train_No']} ({int(max_row['Num_Stops'])} stops)")
print(f"Train with FEWEST stops: Train {min_row['Train_No']} ({int(min_row['Num_Stops'])} stops)")

# Bar chart — stop count distribution
fig, ax = plt.subplots()
stops_per_train['Num_Stops'].clip(upper=50).hist(bins=40, color='steelblue', edgecolor='white', ax=ax)
ax.set_title('Distribution of Stops per Train (clipped at 50)', fontsize=13)
ax.set_xlabel('Number of Stops')
ax.set_ylabel('Number of Trains')
plt.tight_layout()
plt.show()

---
## Level 2: Simple Data Processing
> Goal: Standardize times, compute journey durations, classify routes, station frequency.

In [ ]:
# Task 2.1 — Standardize arrival and departure times
df['Arrival_time'] = pd.to_datetime(df['Arrival_time'], format='%H:%M:%S', errors='coerce')
df['Departure_Time'] = pd.to_datetime(df['Departure_Time'], format='%H:%M:%S', errors='coerce')

print("Time columns parsed to datetime.")
print(f"Null Arrival times  : {df['Arrival_time'].isna().sum()}")
print(f"Null Departure times: {df['Departure_Time'].isna().sum()}")
df[['Train_No','Station_Name','Arrival_time','Departure_Time']].head(5)

In [ ]:
# Task 2.2 — Total journey duration per train
first_dep = df.groupby('Train_No')['Departure_Time'].first()
last_arr  = df.groupby('Train_No')['Arrival_time'].last()
max_dist  = df.groupby('Train_No')['Distance'].max()

duration_hrs = ((last_arr - first_dep).dt.total_seconds() / 3600).abs()

journey_df = pd.DataFrame({
    'Total_Distance_km': max_dist,
    'Journey_Duration_hrs': duration_hrs.round(2)
}).reset_index()

print("Journey durations computed.")
journey_df.head(10)

In [ ]:
# Task 2.3 — Classify routes as Short / Medium / Long
bins   = [0, 300, 800, float('inf')]
labels = ['Short', 'Medium', 'Long']
journey_df['Route_Type'] = pd.cut(journey_df['Total_Distance_km'], bins=bins, labels=labels)

print("Route classification:")
print(journey_df['Route_Type'].value_counts())

fig, ax = plt.subplots(figsize=(6,5))
journey_df['Route_Type'].value_counts().plot(kind='bar', color=['#4caf50','#ff9800','#f44336'],
                                              edgecolor='white', ax=ax)
ax.set_title('Route Type Distribution')
ax.set_xlabel('Route Type')
ax.set_ylabel('Number of Trains')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Task 2.4 — Station-wise train frequency
station_freq = df['Station_Name'].value_counts().reset_index()
station_freq.columns = ['Station_Name', 'Train_Count']

print("Top 15 stations by train frequency:")
print(station_freq.head(15).to_string(index=False))

---
## Level 3: Data Quality Checks
> Goal: Handle missing values, remove duplicates, verify station order, save clean dataset.

In [ ]:
# Task 3.1 — Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Task 3.2 — Remove duplicate Train + Station records
before = len(df)
df_clean = df.drop_duplicates(subset=['Train_No', 'Station_Code'])
after = len(df_clean)
print(f"Rows before deduplication: {before:,}")
print(f"Rows after  deduplication: {after:,}")
print(f"Duplicates removed       : {before - after}")

In [ ]:
# Task 3.3 — Verify correct station order (SN should be ascending per train)
order_issues = (
    df_clean.groupby('Train_No')['SN']
    .apply(lambda x: (x.diff().dropna() <= 0).sum())
    .sum()
)
print(f"Station ordering issues found: {order_issues}")
if order_issues == 0:
    print("All trains have correct station order.")

In [ ]:
# Task 3.4 — Save verified dataset
df_clean.to_csv('cleaned_dataset.csv', index=False)
print(f"Clean dataset saved: 'cleaned_dataset.csv'  ({len(df_clean):,} rows)")

---
## Level 4: Basic Analysis and Visualization
> Goal: Compare journey durations, identify high-traffic stations, visualize key patterns.

In [ ]:
# Task 4.1 — Average journey duration by route type
df_clean2 = pd.read_csv('cleaned_dataset.csv')
df_clean2['Arrival_time']   = pd.to_datetime(df_clean2['Arrival_time'],   format='%H:%M:%S', errors='coerce')
df_clean2['Departure_Time'] = pd.to_datetime(df_clean2['Departure_Time'], format='%H:%M:%S', errors='coerce')

first_dep2 = df_clean2.groupby('Train_No')['Departure_Time'].first()
last_arr2  = df_clean2.groupby('Train_No')['Arrival_time'].last()
max_dist2  = df_clean2.groupby('Train_No')['Distance'].max()
dur2       = ((last_arr2 - first_dep2).dt.total_seconds() / 3600).abs()

journey2 = pd.DataFrame({
    'Total_Distance_km': max_dist2,
    'Journey_Duration_hrs': dur2.round(2)
}).reset_index()
journey2['Route_Type'] = pd.cut(journey2['Total_Distance_km'],
                                 bins=[0,300,800,float('inf')],
                                 labels=['Short','Medium','Long'])

avg_dur = journey2.groupby('Route_Type')['Journey_Duration_hrs'].mean().round(2)
print("Average journey duration by route type (hours):")
print(avg_dur)

In [ ]:
# Task 4.2 — High-traffic stations (top 15)
high_traffic = df_clean2['Station_Name'].value_counts().head(15)
print("Top 15 high-traffic stations:")
print(high_traffic.to_string())

In [ ]:
# Task 4.3 — Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Level 4: Journey Duration & Station Traffic Analysis', fontsize=14, fontweight='bold')

# Plot 1: Avg duration by route type
colors = ['#4caf50','#ff9800','#f44336']
avg_dur.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Avg Journey Duration by Route Type')
axes[0].set_xlabel('Route Type')
axes[0].set_ylabel('Hours')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(avg_dur):
    axes[0].text(i, v + 0.1, f'{v:.1f}h', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Top 15 stations (horizontal bar)
high_traffic.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 High-Traffic Stations')
axes[1].set_xlabel('Train Count')
axes[1].invert_yaxis()

# Plot 3: Route type pie
route_counts = journey2['Route_Type'].value_counts()
axes[2].pie(route_counts, labels=route_counts.index, autopct='%1.1f%%',
            colors=['#4caf50','#ff9800','#f44336'], startangle=140)
axes[2].set_title('Route Type Distribution')

plt.tight_layout()
plt.savefig('level4_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: level4_visualization.png")

In [ ]:
# Task 4.4 — Key observations
print("=" * 55)
print("KEY OBSERVATIONS — Level 4")
print("=" * 55)
print(f"1. Short routes have avg duration of {avg_dur['Short']:.1f} hrs.")
print(f"2. Medium routes have avg duration of {avg_dur['Medium']:.1f} hrs.")
print(f"3. Long routes have avg duration of {avg_dur['Long']:.1f} hrs.")
print(f"4. Busiest station: {high_traffic.index[0]} ({high_traffic.iloc[0]:,} trains)")
print(f"5. Short routes dominate: {route_counts['Short']:,} trains ({route_counts['Short']/route_counts.sum()*100:.1f}%)")
print("=" * 55)

---
## Level 5: Advanced Analysis and Visualization
> Goal: Pivot tables, cross-tabulations, comparative charts.

In [ ]:
# Task 5.1 — Pivot table: station-wise train distribution by route type
df_merged = df_clean2.merge(
    journey2[['Train_No','Route_Type']],
    on='Train_No', how='left'
)

top10_stations = df_clean2['Station_Name'].value_counts().head(10).index
df_top = df_merged[df_merged['Station_Name'].isin(top10_stations)]

pivot = df_top.groupby(['Station_Name','Route_Type']).size().unstack(fill_value=0)
print("Pivot Table — Station × Route Type:")
print(pivot)

In [ ]:
# Task 5.2 — Cross-tabulation: Route Number vs Route Type
crosstab = pd.crosstab(df_merged['Route_Number'], df_merged['Route_Type'])
print("Cross-tabulation — Route Number × Route Type:")
print(crosstab)

In [ ]:
# Task 5.3 — Visualize pivot and cross-tab results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Level 5: Advanced Analysis — Station × Route Type', fontsize=14, fontweight='bold')

# Heatmap of pivot
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Heatmap: Station × Route Type Train Count')
axes[0].set_xlabel('Route Type')
axes[0].set_ylabel('Station')

# Stacked bar of pivot
pivot.plot(kind='bar', stacked=True, ax=axes[1],
           color=['#4caf50','#ff9800','#f44336'], edgecolor='white')
axes[1].set_title('Stacked Bar: Train Count per Station by Route Type')
axes[1].set_xlabel('Station')
axes[1].set_ylabel('Train Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Route Type', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.savefig('level5_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: level5_visualization.png")

In [ ]:
# Task 5.4 — Advanced insights summary
print("=" * 60)
print("ADVANCED INSIGHTS — Level 5")
print("=" * 60)
for station in pivot.index:
    row = pivot.loc[station]
    total = row.sum()
    dom  = row.idxmax()
    print(f"{station:<20} | Total: {total:4d} | Dominant: {dom}")
print("=" * 60)
print(f"Longest route in dataset : {journey2['Total_Distance_km'].max():.0f} km")
print(f"Train with most stops    : {stops_per_train.loc[stops_per_train['Num_Stops'].idxmax(), 'Train_No']} ({stops_per_train['Num_Stops'].max()} stops)")
print("=" * 60)

---
## Level 6: Final Capstone — Interactive Train Enquiry System
> Goal: Build an interactive system to find direct trains, journey duration between any two stations.

In [ ]:
# Build route lookup index from clean dataset
route_index = {}
for train_no, group in df_clean2.groupby('Train_No'):
    group_sorted = group.sort_values('SN')
    route_index[train_no] = group_sorted[['Station_Name','Arrival_time','Departure_Time','Distance']].values.tolist()

print(f"Route index built for {len(route_index):,} trains.")

In [ ]:
def time_diff_str(t1, t2):
    """Compute duration between two time strings HH:MM:SS, handling overnight."""
    try:
        fmt = "%H:%M:%S"
        from datetime import datetime
        d1 = datetime.strptime(str(t1)[:8], fmt) if len(str(t1)) >= 5 else None
        d2 = datetime.strptime(str(t2)[:8], fmt) if len(str(t2)) >= 5 else None
        if d1 is None or d2 is None:
            return "N/A"
        diff = (d2 - d1).total_seconds()
        if diff < 0:
            diff += 86400
        h = int(diff // 3600)
        m = int((diff % 3600) // 60)
        return f"{h}h {m}m"
    except:
        return "N/A"


def enquiry(source, destination):
    """Find all direct trains between source and destination."""
    src = source.strip().upper()
    dst = destination.strip().upper()

    results = []
    for train_no, stops in route_index.items():
        stations = [s[0] for s in stops]
        if src in stations and dst in stations:
            src_idx = stations.index(src)
            dst_idx = stations.index(dst)
            if src_idx < dst_idx:
                src_stop = stops[src_idx]
                dst_stop = stops[dst_idx]
                dep_time = str(src_stop[2])[:8] if src_stop[2] else "N/A"
                arr_time = str(dst_stop[1])[:8] if dst_stop[1] else "N/A"
                dist_km  = int(dst_stop[3]) - int(src_stop[3])
                duration = time_diff_str(dep_time, arr_time)
                results.append({
                    'Train_No'   : train_no,
                    'From'       : src,
                    'Departs'    : dep_time,
                    'To'         : dst,
                    'Arrives'    : arr_time,
                    'Duration'   : duration,
                    'Distance_km': dist_km,
                    'Stops_btwn' : dst_idx - src_idx - 1
                })

    if not results:
        print(f"No direct trains found between '{src}' and '{dst}'.")
        return None

    result_df = pd.DataFrame(results).sort_values('Departs').reset_index(drop=True)
    print(f"\n{'='*70}")
    print(f"  Direct trains: {src}  →  {dst}")
    print(f"  {len(result_df)} train(s) found")
    print(f"{'='*70}")
    print(result_df.to_string(index=False))
    print(f"{'='*70}")
    return result_df


print("Enquiry function ready.")
print("Usage: enquiry('SOURCE STATION', 'DESTINATION STATION')")

In [ ]:
# --- Sample Query 1 ---
results1 = enquiry('HOWRAH JN.', 'BARDDHAMAN J')

In [ ]:
# --- Sample Query 2 ---
results2 = enquiry('CST-MUMBAI', 'THANE')

In [ ]:
# --- Sample Query 3 ---
results3 = enquiry('NEW DELHI', 'AGRA CANTT')

In [ ]:
# --- Interactive input (uncomment to use in Jupyter) ---
# source = input("Enter Source Station: ").strip().upper()
# destination = input("Enter Destination Station: ").strip().upper()
# enquiry(source, destination)

---
## Project Summary

| Level | Title | Key Output |
|-------|-------|-----------|
| 1 | Basic Data Review | 11,113 trains, 8,099 stations, 186,044 records |
| 2 | Data Processing | Durations computed, routes classified (Short/Medium/Long) |
| 3 | Quality Checks | 30 duplicates removed, 0 ordering errors, clean CSV saved |
| 4 | Analysis & Viz | Avg durations, top stations chart, `level4_visualization.png` |
| 5 | Advanced Analysis | Pivot heatmap, stacked bar, `level5_visualization.png` |
| 6 | Capstone Enquiry | Interactive `enquiry()` function for any source → destination |

> **Sysslan IT Solutions — Internship Cohort** · Train Schedule Analysis Using Python